# 05 — Error analysis

This notebook audits the saved test predictions: confusion matrices, bacterial-vs-viral errors, paired bootstrap, calibration, misclassified examples, and Grad-CAM. All analysis uses saved files; nothing is retrained.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import pandas as pd
base = pd.read_csv("results/baseline/seed42/predictions_test.csv")
prop = pd.read_csv("results/proposed/seed42/predictions_test.csv")
print("baseline macro errors:", (base.true_label != base.predicted_label).sum(), "/", len(base))
print("proposed macro errors:", (prop.true_label != prop.predicted_label).sum(), "/", len(prop))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

names = ["NORMAL", "BACTERIA", "VIRUS"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (title, df) in zip(axes, [("baseline", base), ("proposed", prop)]):
    cm = confusion_matrix(df.true_label, df.predicted_label, labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.tight_layout()
plt.show()

In [ ]:
import json
print(json.dumps(json.load(open("results/bacterial_viral_confusion.json")), indent=2))

In [ ]:
print(json.dumps(json.load(open("results/bootstrap_comparison.json")), indent=2)[:3000])

In [ ]:
from IPython.display import Image, display
for name in (
    "confusion_matrices_baseline_vs_proposed.png",
    "bacterial_viral_confusion.png",
    "per_class_f1.png",
    "reliability_diagram.png",
    "proposed_misclassified_test.png",
    "gradcam_proposed_test.png",
):
    display(Image(filename=f"results/figures/{name}"))

## Interpretation

- The proposed model reduces total test errors and substantially reduces viral-to-bacterial confusion.
- VIRUS remains the lowest-F1 class for both models; bacterial and viral pneumonia can look similar on pediatric radiographs.
- AUROC is high while thresholded macro-F1 is lower, especially for the baseline. Calibration is moderate, so probabilities are not clinical risks.
- Grad-CAM shows that some correct predictions attend to lung/mediastinal regions, but several errors focus on borders or text markers. It is a qualitative diagnostic, not a validated localization method.